### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="churn",
    dataset_year="2005",
    domain_str="technology & internet",
    # Data Source
    dataset_source="OpenML",
    original_dataset_source_download_link="https://github.com/EpistasisLab/pmlb/tree/master/datasets/churn", # also https://www.openml.org/d/40701
    download_description="""
curl -L -o churn.tsv.gz "https://media.githubusercontent.com/media/EpistasisLab/pmlb/master/datasets/churn/churn.tsv.gz" \
&& mkdir -p local-data-warehouse/churn/ \
&& mv churn.tsv.gz local-data-warehouse/churn/ \
&& gzip -d local-data-warehouse/churn/churn.tsv.gz
""",
    # References
    academic_reference_bibtex="""@misc{marcoulides2005churn,
  title={Discovering knowledge in data: An introduction to data mining},
  author={Marcoulides, George A},
  year={2005},
  publisher={Taylor \& Francis}
}
""",
    academic_reference_bibtex_key="marcoulides2005churn",
    license="Public Domain",
    data_tags=["IID", "Spatial"],
    curation_comments="""
- The original source is lost, so we use https://github.com/EpistasisLab/pmlb/tree/master/datasets/churn (or https://www.openml.org/d/40701)
- We dropped the "phone_number" feature as it seems to be an index in the original data. 
- We renamed the target variable to "CustomerChurned" and mapped binary variables to "Yes"/"No"
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="CustomerChurned",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="CustomerChurned",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/churn.tsv", sep="\t")
df.columns = df.columns.str.replace(" ", "_")

df.rename(columns={"target": "CustomerChurned"}, inplace=True)
df.drop(columns=["phone_number"], inplace=True)

cat_features = [
    "state",
    "area_code",
    "international_plan",
    "voice_mail_plan",
    "CustomerChurned"
]

df["CustomerChurned"] = df["CustomerChurned"].map({1: "Yes", 0: "No"})
df["international_plan"] = df["international_plan"].map({1: "Yes", 0: "No"})
df["voice_mail_plan"] = df["voice_mail_plan"].map({1: "Yes", 0: "No"})

df[cat_features] = df[cat_features].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()